In [2]:
import transformers
from transformers import AutoConfig

print("Transformers:", transformers.__version__)

for model_id in ["Qwen/Qwen3-8B", "Qwen/Qwen3.5-9B"]:
    try:
        config = AutoConfig.from_pretrained(model_id)
        print(f"\n{model_id}")
        print("Model type:", config.model_type)
        print("Architectures:", getattr(config, "architectures", None))
    except Exception as e:
        print(f"\n{model_id} config check failed: {type(e).__name__}: {e}")


Transformers: 5.16.1


config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]


Qwen/Qwen3-8B
Model type: qwen3
Architectures: ['Qwen3ForCausalLM']


config.json:   0%|          | 0.00/3.13k [00:00<?, ?B/s]


Qwen/Qwen3.5-9B
Model type: qwen3_5
Architectures: ['Qwen3_5ForConditionalGeneration']


In [4]:
%pip install -q bitsandbytes



Note: you may need to restart the kernel to use updated packages.


In [5]:
import bitsandbytes as bnb
import torch

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("bitsandbytes:", bnb.__version__)
print("GPU:", torch.cuda.get_device_name(0))


PyTorch: 2.11.0+cu128
CUDA: True
bitsandbytes: 0.50.2
GPU: Tesla T4


In [6]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen3-8B"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quant_config,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True,
)

print("Loaded:", MODEL_ID)
print("GPU allocated (GB):", round(torch.cuda.memory_allocated() / 1e9, 2))


tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


... (2 outputs left out)

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Loaded: Qwen/Qwen3-8B
GPU allocated (GB): 1.54


In [7]:
messages = [
    {"role": "user", "content": 'Return JSON only: {"supported": true, "reason": "short reason"}'}
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False,
)

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.inference_mode():
    output = model.generate(
        **inputs,
        max_new_tokens=80,
        do_sample=False,
    )

new_tokens = output[0][inputs["input_ids"].shape[1] :]
print(tokenizer.decode(new_tokens, skip_special_tokens=True))


{"supported": true, "reason": "short reason"}


In [8]:
import gc

import torch

del model, tokenizer

gc.collect()
torch.cuda.empty_cache()

print("GPU allocated (GB):", round(torch.cuda.memory_allocated() / 1e9, 2))
print("GPU reserved (GB):", round(torch.cuda.memory_reserved() / 1e9, 2))


GPU allocated (GB): 0.01
GPU reserved (GB): 1.54


In [9]:
import torch
from transformers import AutoModelForImageTextToText, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen3.5-9B"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    quantization_config=quant_config,
    dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True,
)

print("Loaded:", MODEL_ID)
print("GPU allocated (GB):", round(torch.cuda.memory_allocated() / 1e9, 2))


tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/6.72M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/3.35M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 12.8MB            

tokenizer.json: downloading bytes:           |  0.00B            

chat_template.jinja:   0%|          | 0.00/7.76k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/79.7k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/760 [00:00<?, ?it/s]

Loaded: Qwen/Qwen3.5-9B
GPU allocated (GB): 0.25


In [10]:
messages = [
    {"role": "user", "content": 'Return JSON only: {"supported": true, "reason": "short reason"}'}
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False,
)

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.inference_mode():
    output = model.generate(
        **inputs,
        max_new_tokens=80,
        do_sample=False,
    )

new_tokens = output[0][inputs["input_ids"].shape[1] :]
print(tokenizer.decode(new_tokens, skip_special_tokens=True))


{"supported": true, "reason": "I am an AI assistant capable of generating JSON."}



In [11]:
import json
import time

import torch

print("Model:", MODEL_ID)
print("GPU:", torch.cuda.get_device_name(0))
print("Transformers:", __import__("transformers").__version__)

test_output = tokenizer.decode(new_tokens, skip_special_tokens=True)

try:
    parsed = json.loads(test_output)
    print("Strict JSON validation: PASS")
    print("Parsed value:", parsed)
except json.JSONDecodeError as exc:
    print("Strict JSON validation: FAIL")
    print("Error:", exc)


Model: Qwen/Qwen3.5-9B
GPU: Tesla T4
Transformers: 5.16.1
Strict JSON validation: PASS
Parsed value: {'supported': True, 'reason': 'I am an AI assistant capable of generating JSON.'}


In [12]:
import json
from collections import Counter
from pathlib import Path

cases = []


def add(task, case_id, prompt, gold, scoring_rule):
    cases.append(
        {
            "id": case_id,
            "task": task,
            "prompt": prompt,
            "gold": gold,
            "scoring_rule": scoring_rule,
        }
    )


# Task 1: Extraction — extract the stated relationship.
for i in range(20):
    subject = f"Company_{i+1}"
    object_ = f"Product_{i+1}"
    evidence = f"{subject} launched {object_} in 202{i % 10}."
    add(
        "extraction",
        f"ext_{i+1:02d}",
        f"Extract the subject, relation, object, and year from this evidence. "
        f'Return JSON only with keys subject, relation, object, year.\nEvidence: "{evidence}"',
        {"subject": subject, "relation": "launched", "object": object_, "year": f"202{i % 10}"},
        "exact_json_fields",
    )

# Task 2: Strength assessment — distinguish explicit evidence from weak inference.
for i in range(20):
    claim = f"Company_{i+1} launched Product_{i+1}."
    evidence = claim if i % 2 == 0 else f"Company_{i+1} discussed Product_{i+1}."
    gold_strength = "supported" if i % 2 == 0 else "not_supported"
    add(
        "strength",
        f"str_{i+1:02d}",
        f"Assess whether the evidence supports the claim. Return JSON only with keys "
        f'strength and supported.\nClaim: "{claim}"\nEvidence: "{evidence}"',
        {"strength": gold_strength, "supported": i % 2 == 0},
        "exact_json_fields",
    )

# Task 3: JSON validity — produce a required schema.
for i in range(20):
    add(
        "json_validity",
        f"json_{i+1:02d}",
        f'Return a JSON object only, with keys "id" (integer), "valid" (boolean), '
        f'and "label" (string). Use id={i+1}, valid=true, label="case_{i+1}".',
        {"id": i + 1, "valid": True, "label": f"case_{i+1}"},
        "strict_json_schema",
    )

# Task 4: Tool choice — choose the tool that matches the goal.
tool_cases = [
    ("Find documents mentioning a company", "search_corpus"),
    ("Retrieve a time series for a named indicator", "get_series"),
    ("Check whether a claim is supported by a cited source", "verify_edge"),
    ("Search for evidence against a proposed claim", "counter_search"),
    ("Update the graph after evidence is verified", "update_graph"),
    ("Finish because no work remains", "stop"),
]
for i in range(20):
    goal, tool = tool_cases[i % len(tool_cases)]
    add(
        "tool_choice",
        f"tool_{i+1:02d}",
        f"Choose exactly one tool for this goal. Available tools: search_corpus, "
        f"get_series, verify_edge, counter_search, update_graph, stop. "
        f'Return JSON only with key "tool".\nGoal: {goal}.',
        {"tool": tool},
        "exact_tool",
    )

# Task 5: Unsupported edges — reject claims that exceed the evidence.
for i in range(20):
    claim = f"Company_{i+1} acquired Company_{i+21}."
    evidence = f"Company_{i+1} and Company_{i+21} attended the same conference."
    add(
        "unsupported_edges",
        f"edge_{i+1:02d}",
        f"Is the proposed relationship supported by the evidence? Return JSON only "
        f'with keys supported and decision.\nProposed edge: "{claim}"\n'
        f'Evidence: "{evidence}"',
        {"supported": False, "decision": "reject"},
        "unsupported_edge_rejection",
    )

assert len(cases) == 100
assert len({c["id"] for c in cases}) == 100

out = Path("pilot_cases.jsonl")
out.write_text("".join(json.dumps(c, ensure_ascii=False) + "\n" for c in cases), encoding="utf-8")

print("Saved:", out.resolve())
print("Total cases:", len(cases))
print("Cases per task:", dict(Counter(c["task"] for c in cases)))
print("First case:", json.dumps(cases[0], ensure_ascii=False, indent=2))


Saved: /kaggle/working/pilot_cases.jsonl
Total cases: 100
Cases per task: {'extraction': 20, 'strength': 20, 'json_validity': 20, 'tool_choice': 20, 'unsupported_edges': 20}
First case: {
  "id": "ext_01",
  "task": "extraction",
  "prompt": "Extract the subject, relation, object, and year from this evidence. Return JSON only with keys subject, relation, object, year.\nEvidence: \"Company_1 launched Product_1 in 2020.\"",
  "gold": {
    "subject": "Company_1",
    "relation": "launched",
    "object": "Product_1",
    "year": "2020"
  },
  "scoring_rule": "exact_json_fields"
}


In [13]:
import json
from collections import Counter

with open("/kaggle/working/pilot_cases.jsonl", encoding="utf-8") as f:
    cases = [json.loads(line) for line in f if line.strip()]

print("Total cases:", len(cases))
print("Unique IDs:", len({c["id"] for c in cases}))
print("Unique prompts:", len({c["prompt"] for c in cases}))
print("\nTask counts:", dict(Counter(c["task"] for c in cases)))

print("\n========== SAMPLE CASES ==========")
for task in sorted({c["task"] for c in cases}):
    print(f"\n--- {task} ---")
    samples = [c for c in cases if c["task"] == task][:3]
    for case in samples:
        print(case["id"], ":", case["prompt"][:250].replace("\n", " | "))


Total cases: 100
Unique IDs: 100
Unique prompts: 86

Task counts: {'extraction': 20, 'strength': 20, 'json_validity': 20, 'tool_choice': 20, 'unsupported_edges': 20}

========== SAMPLE CASES ==========

--- extraction ---
ext_01 : Extract the subject, relation, object, and year from this evidence. Return JSON only with keys subject, relation, object, year. | Evidence: "Company_1 launched Product_1 in 2020."
ext_02 : Extract the subject, relation, object, and year from this evidence. Return JSON only with keys subject, relation, object, year. | Evidence: "Company_2 launched Product_2 in 2021."
ext_03 : Extract the subject, relation, object, and year from this evidence. Return JSON only with keys subject, relation, object, year. | Evidence: "Company_3 launched Product_3 in 2022."

--- json_validity ---
json_01 : Return a JSON object only, with keys "id" (integer), "valid" (boolean), and "label" (string). Use id=1, valid=true, label="case_1".
json_02 : Return a JSON object only, with keys

In [14]:
# C2 MODEL PILOT — END-TO-END
# Qwen3-8B vs Qwen3.5-9B
# Outputs: pilot_cases_v2.jsonl, pilot_results.jsonl,
#          pilot_summary.json, pilot_model.md

import gc
import hashlib
import json
import platform
from collections import Counter, defaultdict
from datetime import UTC, datetime
from pathlib import Path

import torch
import transformers
from transformers import (
    AutoModelForCausalLM,
    AutoModelForImageTextToText,
    AutoTokenizer,
    BitsAndBytesConfig,
)

OUT = Path("/kaggle/working")
CASE_FILE = OUT / "pilot_cases_v2.jsonl"
RESULT_FILE = OUT / "pilot_results.jsonl"
SUMMARY_FILE = OUT / "pilot_summary.json"
REPORT_FILE = OUT / "pilot_model.md"

assert torch.cuda.is_available(), "Enable a Kaggle GPU before running."
assert transformers.__version__.startswith(
    "5."
), f"Expected the tested Transformers 5.x environment, got {transformers.__version__}"

# ---------- 1. Frozen evaluation cases ----------
cases = []


def add(task, cid, prompt, gold, rule):
    cases.append({"id": cid, "task": task, "prompt": prompt, "gold": gold, "scoring_rule": rule})


# Extraction: varied entities, relations, dates, and phrasing.
extractions = [
    (
        "In March 2021, Northstar Ltd acquired a majority stake in Elm Robotics.",
        "Northstar Ltd",
        "acquired a majority stake in",
        "Elm Robotics",
        "March 2021",
    ),
    (
        "Aster Labs opened its first Berlin office during September 2019.",
        "Aster Labs",
        "opened",
        "its first Berlin office",
        "September 2019",
    ),
    (
        "The regulator approved NovaMed's device on 14 June 2022.",
        "The regulator",
        "approved",
        "NovaMed's device",
        "14 June 2022",
    ),
    (
        "On 3 January 2020, Cedar Bank appointed Mira Shah as chief risk officer.",
        "Cedar Bank",
        "appointed",
        "Mira Shah as chief risk officer",
        "3 January 2020",
    ),
    (
        "Orion Energy signed a five-year supply agreement with Delta Works in 2018.",
        "Orion Energy",
        "signed",
        "a five-year supply agreement with Delta Works",
        "2018",
    ),
    (
        "The board rejected the merger proposal on 22 April 2023.",
        "The board",
        "rejected",
        "the merger proposal",
        "22 April 2023",
    ),
    (
        "In November 2017, Kestrel Systems launched its Atlas platform.",
        "Kestrel Systems",
        "launched",
        "its Atlas platform",
        "November 2017",
    ),
    (
        "Pine Health discontinued the CardioPlus service in July 2024.",
        "Pine Health",
        "discontinued",
        "the CardioPlus service",
        "July 2024",
    ),
    (
        "A filing dated 5 May 2020 says that Lumen Group sold its Chilean unit to Arco SA.",
        "Lumen Group",
        "sold",
        "its Chilean unit to Arco SA",
        "5 May 2020",
    ),
    (
        "By February 2016, Westbridge University had partnered with Meridian AI.",
        "Westbridge University",
        "had partnered with",
        "Meridian AI",
        "February 2016",
    ),
    (
        "On 18 August 2021, Solace Foods reported a 12% increase in annual revenue.",
        "Solace Foods",
        "reported",
        "a 12% increase in annual revenue",
        "18 August 2021",
    ),
    (
        "The court suspended the licence of Harbor Mining in December 2022.",
        "The court",
        "suspended",
        "the licence of Harbor Mining",
        "December 2022",
    ),
    (
        "In 2015, Redfern Motors began manufacturing electric buses in Pune.",
        "Redfern Motors",
        "began manufacturing",
        "electric buses in Pune",
        "2015",
    ),
    (
        "A notice issued on 9 October 2023 extended the deadline for BluePeak's filing.",
        "A notice",
        "extended",
        "the deadline for BluePeak's filing",
        "9 October 2023",
    ),
    (
        "Cobalt Telecom transferred 40 spectrum licences to Vela Mobile in 2020.",
        "Cobalt Telecom",
        "transferred",
        "40 spectrum licences to Vela Mobile",
        "2020",
    ),
    (
        "In April 2018, the agency fined Summit Chemicals for a reporting violation.",
        "the agency",
        "fined",
        "Summit Chemicals for a reporting violation",
        "April 2018",
    ),
    (
        "The annual report states that Fern Retail entered the Kenyan market in 2022.",
        "Fern Retail",
        "entered",
        "the Kenyan market",
        "2022",
    ),
    (
        "On 11 December 2019, Axis Rail awarded the construction contract to BuildCore.",
        "Axis Rail",
        "awarded",
        "the construction contract to BuildCore",
        "11 December 2019",
    ),
    (
        "A 2024 notice says that the ministry withdrew approval for the Rivergate project.",
        "the ministry",
        "withdrew",
        "approval for the Rivergate project",
        "2024",
    ),
    (
        "By June 2020, Beacon Analytics had acquired a 30% interest in Quanta Data.",
        "Beacon Analytics",
        "had acquired",
        "a 30% interest in Quanta Data",
        "June 2020",
    ),
]
for i, (text, s, r, o, t) in enumerate(extractions, 1):
    add(
        "extraction",
        f"ext_{i:02}",
        "Extract the subject, relation, object, and time exactly as stated. "
        "Return one JSON object only, with string fields subject, relation, object, time. "
        "Do not add facts.\nEvidence: " + text,
        {"subject": s, "relation": r, "object": o, "time": t},
        "exact_fields",
    )

# Strength: direct support, partial support, contradiction, insufficient evidence.
strengths = [
    (
        "A report says Acme opened a factory in 2020.",
        "Acme opened a factory in 2020.",
        "direct_support",
    ),
    (
        "A memo says Acme considered opening a factory.",
        "Acme opened a factory.",
        "insufficient_evidence",
    ),
    (
        "The filing says the deal was cancelled before signing.",
        "The deal was completed.",
        "contradicted",
    ),
    ("The report confirms revenue rose 8%.", "Revenue rose.", "direct_support"),
    ("The report says revenue rose in Europe.", "Revenue rose globally.", "partial_support"),
    (
        "The company announced a pilot programme.",
        "The company launched a nationwide service.",
        "insufficient_evidence",
    ),
    (
        "The regulator denied that a permit had been issued.",
        "A permit had been issued.",
        "contradicted",
    ),
    (
        "The statement names Lee as the newly appointed CFO.",
        "Lee was appointed CFO.",
        "direct_support",
    ),
    (
        "The source says the firm explored a possible acquisition.",
        "The firm acquired its competitor.",
        "insufficient_evidence",
    ),
    (
        "The source says output increased from 100 to 120 units.",
        "Output increased by 20 units.",
        "direct_support",
    ),
    (
        "The source confirms a contract with Vendor A, not Vendor B.",
        "The company contracted with Vendor B.",
        "contradicted",
    ),
    (
        "The company announced plans to enter Spain.",
        "The company operates in Spain.",
        "partial_support",
    ),
    (
        "The article reports a meeting between the two firms.",
        "The firms merged.",
        "insufficient_evidence",
    ),
    ("The audit states that no payment was made.", "The payment was made.", "contradicted"),
    (
        "The report confirms the project began in 2021.",
        "The project began in 2021.",
        "direct_support",
    ),
    (
        "The filing says a product was tested in one hospital.",
        "The product is proven effective in all hospitals.",
        "partial_support",
    ),
    (
        "The source mentions an allegation but gives no findings.",
        "The allegation was proven true.",
        "insufficient_evidence",
    ),
    ("The notice says the licence remains active.", "The licence was revoked.", "contradicted"),
    (
        "The report states that 15 branches opened in 2023.",
        "Fifteen branches opened in 2023.",
        "direct_support",
    ),
    (
        "The company signed a non-binding letter of intent.",
        "The acquisition was finalised.",
        "insufficient_evidence",
    ),
]
for i, (evidence, claim, strength) in enumerate(strengths, 1):
    supported = strength in ("direct_support", "partial_support")
    add(
        "strength",
        f"str_{i:02}",
        "Assess the evidence against the claim. Use strength exactly one of "
        "direct_support, partial_support, contradicted, insufficient_evidence. "
        "Set supported=true only for direct_support or partial_support; otherwise false. "
        "Return JSON only with keys strength (string) and supported (boolean).\n"
        f"Evidence: {evidence}\nClaim: {claim}",
        {"strength": strength, "supported": supported},
        "exact_fields",
    )

# JSON validity: varied required schemas and types.
json_specs = [
    ({"id": 1, "active": True, "name": "alpha"}),
    ({"count": 7, "ratio": 0.25, "label": "beta"}),
    ({"items": ["red", "blue"], "ok": True}),
    ({"meta": {"source": "filing", "year": 2022}, "valid": True}),
    ({"code": "E4", "attempts": 3, "passed": False}),
    ({"values": [2, 4, 6], "name": "gamma"}),
    ({"record": {"id": 9, "tag": "x"}, "enabled": False}),
    ({"score": 0.91, "rank": 2, "model": "qwen"}),
    ({"dates": ["2020-01-01", "2021-01-01"], "count": 2}),
    ({"entity": "Northstar", "relation": "acquired", "year": 2021}),
    ({"status": "rejected", "supported": False}),
    ({"bounds": {"low": 1.5, "high": 3.5}, "unit": "kg"}),
    ({"ids": [1, 3, 5], "complete": True}),
    ({"result": {"ok": True, "message": "done"}, "version": 1}),
    ({"name": "delta", "aliases": ["d", "D"], "active": True}),
    ({"probability": 0.6, "threshold": 0.5, "accepted": True}),
    ({"source": {"title": "Report A", "pages": 12}, "verified": False}),
    ({"edges": [{"from": "A", "to": "B"}], "total": 1}),
    ({"currency": "USD", "amount": 125.75, "refunded": False}),
    ({"label": "final", "notes": None, "sequence": 4}),
]


def describe(value):
    if value is None:
        return "null"
    if type(value) is bool:
        return "boolean"
    if type(value) is int:
        return "integer"
    if type(value) is float:
        return "number"
    if type(value) is str:
        return "string"
    if type(value) is list:
        return "array; elements must match this example's element types"
    if type(value) is dict:
        return "object with exactly these nested keys and value types"
    return type(value).__name__


for i, expected in enumerate(json_specs, 1):
    add(
        "json_validity",
        f"json_{i:02}",
        "Return JSON only, without Markdown fences or commentary. The JSON must contain "
        "exactly the keys and values shown in this specification, with the same JSON types. "
        "Do not omit keys or add keys.\nSpecification:\n"
        + json.dumps(expected, ensure_ascii=False),
        expected,
        "strict_json_and_schema",
    )

# Tool choice: each available tool is exercised in distinct contexts.
tool_cases = [
    ("Find corpus documents mentioning a named company.", "search_corpus"),
    ("Retrieve quarterly GDP values for a named country and date range.", "get_series"),
    ("Check whether a proposed claim is supported by its cited source.", "verify_edge"),
    ("Search for evidence that could disprove the proposed acquisition claim.", "counter_search"),
    ("Add a new graph relationship after it has been verified.", "update_graph"),
    ("The task is complete and no further action is needed.", "stop"),
    ("Find articles containing the phrase 'supply disruption'.", "search_corpus"),
    ("Fetch the unemployment-rate series from 2019 through 2023.", "get_series"),
    ("Inspect a source span to decide whether it supports a specific edge.", "verify_edge"),
    ("Look for contradictory reporting about the reported merger.", "counter_search"),
    ("Persist a verified entity relationship in the knowledge graph.", "update_graph"),
    ("No open edges remain and the stopping criteria are met.", "stop"),
    ("Search indexed reports for the named regulator.", "search_corpus"),
    ("Retrieve monthly inflation observations for the requested period.", "get_series"),
    ("Validate the evidence attached to a candidate ownership relation.", "verify_edge"),
    ("Find sources challenging the claim that the permit was granted.", "counter_search"),
    ("Commit the supported and verified relationship to the graph.", "update_graph"),
    ("The budget is exhausted; terminate the agent run.", "stop"),
    ("Search the corpus for the company's annual report.", "search_corpus"),
    ("Get the historical exchange-rate series for the specified dates.", "get_series"),
]
for i, (goal, tool) in enumerate(tool_cases, 1):
    add(
        "tool_choice",
        f"tool_{i:02}",
        "Select exactly one tool from: search_corpus, get_series, verify_edge, "
        "counter_search, update_graph, stop. Return JSON only with key tool.\nGoal: " + goal,
        {"tool": tool},
        "exact_tool",
    )

# Unsupported edges: co-occurrence, plans, roles, timing, and correlation aren't proof.
edges = [
    ("Apex acquired Birch.", "Apex and Birch sponsored the same conference."),
    ("Mira owns 60% of Koru Ltd.", "Mira attended a Koru Ltd investor presentation."),
    (
        "Zenith caused the outage.",
        "Zenith's product was used by a customer who experienced an outage.",
    ),
    ("The two firms merged.", "The firms announced exploratory merger talks."),
    ("Nile Bank funded Orion Ltd.", "Nile Bank and Orion Ltd share a board adviser."),
    ("The medicine cures the disease.", "A small uncontrolled study reported symptom changes."),
    ("The minister approved the contract.", "The minister met executives from the contractor."),
    (
        "The company committed fraud.",
        "An anonymous post alleges fraud; no corroboration is provided.",
    ),
    ("The product was recalled.", "The regulator opened a safety review."),
    (
        "Company A acquired Company B in 2022.",
        "A 2024 article says the companies discussed a partnership.",
    ),
    ("The CEO resigned.", "The CEO's biography was removed from the website."),
    (
        "The policy reduced unemployment.",
        "Unemployment fell after the policy; no causal analysis is provided.",
    ),
    (
        "The supplier breached the contract.",
        "Delivery was delayed, but the contract's terms are not supplied.",
    ),
    ("The new drug is safer than Drug X.", "A trial compared the new drug only with placebo."),
    (
        "The subsidiary is wholly owned by ParentCo.",
        "Both entities use the same registered address.",
    ),
    (
        "The company concealed its losses.",
        "The company restated an accounting figure; intent is not established.",
    ),
    ("The project received final approval.", "The project was submitted for approval."),
    ("The device is certified for children.", "The device is certified for adult use only."),
    ("The two people are related.", "They have the same surname and live in the same city."),
    (
        "The firm violated the law.",
        "A regulator opened an investigation but has issued no finding.",
    ),
]
for i, (claim, evidence) in enumerate(edges, 1):
    add(
        "unsupported_edges",
        f"edge_{i:02}",
        "Decide whether the evidence supports the proposed edge. Mere co-occurrence, "
        "correlation, allegation, discussion, or investigation is not sufficient proof. "
        "Return JSON only with keys supported (boolean) and decision (string). "
        "Use supported=false and decision='reject' when the edge is unsupported.\n"
        f"Proposed edge: {claim}\nEvidence: {evidence}",
        {"supported": False, "decision": "reject"},
        "reject_unsupported",
    )

assert len(cases) == 100
assert len({c["id"] for c in cases}) == 100
assert Counter(c["task"] for c in cases) == {
    "extraction": 20,
    "strength": 20,
    "json_validity": 20,
    "tool_choice": 20,
    "unsupported_edges": 20,
}
assert len({c["prompt"] for c in cases}) == 100

# Save frozen dataset and its checksum.
case_bytes = "".join(
    json.dumps(c, ensure_ascii=False, sort_keys=True) + "\n" for c in cases
).encode("utf-8")
CASE_FILE.write_bytes(case_bytes)
dataset_sha = hashlib.sha256(case_bytes).hexdigest()
print("Frozen dataset:", CASE_FILE)
print("Dataset SHA-256:", dataset_sha)
print("Cases:", len(cases), dict(Counter(c["task"] for c in cases)))


# ---------- 2. Strict scoring ----------
def strict_parse(raw):
    try:
        value = json.loads(raw.strip())
        return value, True
    except Exception:
        return None, False


def schema_type_ok(value, expected):
    if expected is None:
        return value is None
    if type(expected) is bool:
        return type(value) is bool
    if type(expected) is int:
        return type(value) is int
    if type(expected) is float:
        return type(value) in (int, float) and type(value) is not bool
    if type(expected) is str:
        return type(value) is str
    if type(expected) is list:
        return (
            type(value) is list
            and len(value) == len(expected)
            and all(schema_type_ok(v, e) for v, e in zip(value, expected, strict=True))
        )
    if type(expected) is dict:
        return (
            type(value) is dict
            and set(value) == set(expected)
            and all(schema_type_ok(value[k], expected[k]) for k in expected)
        )
    return False


def score_case(case, raw):
    parsed, valid_json = strict_parse(raw)
    result = {
        "strict_json_valid": valid_json,
        "schema_valid": False,
        "exact_match": False,
        "score": 0,
    }
    if not valid_json or not isinstance(parsed, dict):
        return result

    gold, task = case["gold"], case["task"]
    if task == "json_validity":
        result["schema_valid"] = schema_type_ok(parsed, gold)
        result["exact_match"] = parsed == gold
        result["score"] = int(result["schema_valid"])
    elif task == "extraction":
        result["schema_valid"] = set(parsed) == set(gold) and all(
            isinstance(parsed[k], str) for k in gold
        )
        result["exact_match"] = result["schema_valid"] and all(
            parsed[k].strip() == gold[k] for k in gold
        )
        result["score"] = int(result["exact_match"])
    elif task == "strength":
        result["schema_valid"] = (
            set(parsed) == set(gold)
            and isinstance(parsed.get("strength"), str)
            and type(parsed.get("supported")) is bool
        )
        result["exact_match"] = result["schema_valid"] and parsed == gold
        result["score"] = int(result["exact_match"])
    elif task == "tool_choice":
        result["schema_valid"] = set(parsed) == {"tool"} and isinstance(parsed.get("tool"), str)
        result["exact_match"] = result["schema_valid"] and parsed["tool"] == gold["tool"]
        result["score"] = int(result["exact_match"])
    elif task == "unsupported_edges":
        result["schema_valid"] = (
            set(parsed) == set(gold)
            and type(parsed.get("supported")) is bool
            and isinstance(parsed.get("decision"), str)
        )
        result["exact_match"] = result["schema_valid"] and (
            parsed["supported"] is False and parsed["decision"].strip().lower() == "reject"
        )
        result["score"] = int(result["exact_match"])
    return result


# ---------- 3. Run models sequentially ----------
models = [
    ("Qwen/Qwen3-8B", "causal"),
    ("Qwen/Qwen3.5-9B", "multimodal"),
]
quant = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

summary = {
    "created_utc": datetime.now(UTC).isoformat(),
    "python": platform.python_version(),
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "gpu": torch.cuda.get_device_name(0),
    "quantization": "bitsandbytes NF4 4-bit; double quantization enabled",
    "dataset": CASE_FILE.name,
    "dataset_sha256": dataset_sha,
    "cases": len(cases),
    "decoding": {"do_sample": False, "max_new_tokens": 160, "enable_thinking": False},
    "models": {},
}

if RESULT_FILE.exists():
    RESULT_FILE.unlink()

for model_id, architecture in models:
    print(f"\n===== Loading {model_id} =====")
    model = tokenizer = None
    model_info = {"model_id": model_id, "status": "started"}
    try:
        tokenizer = AutoTokenizer.from_pretrained(model_id)
        cls = AutoModelForCausalLM if architecture == "causal" else AutoModelForImageTextToText
        model = cls.from_pretrained(
            model_id,
            quantization_config=quant,
            dtype=torch.float16,
            device_map="auto",
            low_cpu_mem_usage=True,
        )
        model.eval()
        model_info["status"] = "completed"
        model_info["results"] = 0
        print("Loaded; running 100 cases.")

        with RESULT_FILE.open("a", encoding="utf-8") as rf:
            for idx, case in enumerate(cases, 1):
                started = time.perf_counter()
                raw = ""
                error = None
                input_count = output_count = None
                try:
                    messages = [{"role": "user", "content": case["prompt"]}]
                    prompt = tokenizer.apply_chat_template(
                        messages,
                        tokenize=False,
                        add_generation_prompt=True,
                        enable_thinking=False,
                    )
                    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
                    input_count = int(inputs["input_ids"].shape[-1])
                    with torch.inference_mode():
                        generated = model.generate(**inputs, max_new_tokens=160, do_sample=False)
                    new_tokens = generated[0][input_count:]
                    output_count = int(new_tokens.numel())
                    raw = tokenizer.decode(new_tokens, skip_special_tokens=True)
                    scored = score_case(case, raw)
                except Exception as exc:
                    error = f"{type(exc).__name__}: {exc}"
                    scored = {
                        "strict_json_valid": False,
                        "schema_valid": False,
                        "exact_match": False,
                        "score": 0,
                    }

                row = {
                    "model_id": model_id,
                    "case_id": case["id"],
                    "task": case["task"],
                    "raw_output": raw,
                    "gold": case["gold"],
                    "scoring": scored,
                    "input_tokens": input_count,
                    "output_tokens": output_count,
                    "latency_seconds": round(time.perf_counter() - started, 4),
                    "error": error,
                }
                rf.write(json.dumps(row, ensure_ascii=False) + "\n")
                rf.flush()
                model_info["results"] += 1

                if idx % 20 == 0:
                    print(f"{model_id}: {idx}/100 cases")

    except Exception as exc:
        model_info["status"] = "failed"
        model_info["error"] = f"{type(exc).__name__}: {exc}"
        print("MODEL FAILED:", model_info["error"])
    finally:
        if model is not None:
            del model
        if tokenizer is not None:
            del tokenizer
        gc.collect()
        torch.cuda.empty_cache()
        summary["models"][model_id] = model_info

# ---------- 4. Aggregate actual results ----------
result_rows = []
if RESULT_FILE.exists():
    with RESULT_FILE.open(encoding="utf-8") as f:
        result_rows = [json.loads(line) for line in f if line.strip()]

aggregate = defaultdict(lambda: defaultdict(list))
for row in result_rows:
    aggregate[row["model_id"]][row["task"]].append(row)

for model_id in models:
    mid = model_id[0]
    if mid not in summary["models"]:
        summary["models"][mid] = {"status": "not_run"}

    tasks = {}
    for task in ["extraction", "strength", "json_validity", "tool_choice", "unsupported_edges"]:
        rows = aggregate[mid].get(task, [])
        if not rows:
            tasks[task] = {"n": 0, "score": None}
            continue
        tasks[task] = {
            "n": len(rows),
            "score": sum(r["scoring"]["score"] for r in rows) / len(rows),
            "strict_json_valid_rate": sum(r["scoring"]["strict_json_valid"] for r in rows)
            / len(rows),
            "schema_valid_rate": sum(r["scoring"]["schema_valid"] for r in rows) / len(rows),
            "exact_match_rate": sum(r["scoring"]["exact_match"] for r in rows) / len(rows),
            "mean_latency_seconds": sum(r["latency_seconds"] for r in rows) / len(rows),
            "errors": sum(r["error"] is not None for r in rows),
        }
    summary["models"][mid]["tasks"] = tasks
    summary["models"][mid]["completed_results"] = sum(len(v) for v in aggregate[mid].values())

SUMMARY_FILE.write_text(json.dumps(summary, indent=2), encoding="utf-8")

# ---------- 5. Write report from observed results ----------
lines = [
    "# C2 Model Pilot: Qwen3-8B vs Qwen3.5-9B",
    "",
    f"- Run time (UTC): {summary['created_utc']}",
    f"- GPU: {summary['gpu']}",
    f"- Python: {summary['python']}",
    f"- PyTorch: {summary['torch']}",
    f"- Transformers: {summary['transformers']}",
    f"- Quantization: {summary['quantization']}",
    f"- Dataset: `{CASE_FILE.name}`",
    f"- Dataset SHA-256: `{dataset_sha}`",
    "- Design: same frozen prompts, greedy decoding, max_new_tokens=160; models run sequentially.",
    "- No fine-tuning was performed.",
    "",
    "## Results",
    "",
    "| Model | Task | N | Primary score | Strict JSON rate | Schema rate | Exact-match rate | Mean latency (s) | Errors |",
    "|---|---|---:|---:|---:|---:|---:|---:|---:|",
]
for model_id, _ in models:
    mi = summary["models"].get(model_id, {})
    for task, metrics in mi.get("tasks", {}).items():

        def pct(v):
            return "—" if v is None else f"{100*v:.1f}%"

        lines.append(
            f"| {model_id} | {task} | {metrics['n']} | {pct(metrics['score'])} | "
            f"{pct(metrics.get('strict_json_valid_rate'))} | "
            f"{pct(metrics.get('schema_valid_rate'))} | "
            f"{pct(metrics.get('exact_match_rate'))} | "
            f"{metrics.get('mean_latency_seconds', float('nan')):.3f} | "
            f"{metrics.get('errors', 0)} |"
        )
lines += [
    "",
    "## Interpretation and limitations",
    "",
    "- Scores are computed from the recorded raw model outputs using the fixed rules in the notebook.",
    "- The JSON-validity task's primary score is strict schema validity; exact-value accuracy is reported separately.",
    "- The dataset is synthetic and small (20 cases per task); results are exploratory, not a general capability claim.",
    "- Extraction and strength scoring use exact gold labels and may penalize semantically equivalent wording.",
    "- All unsupported-edge cases require rejection; this estimates rejection behavior on these cases, not general calibration.",
    "- Latency depends on Kaggle GPU state, caching, and runtime load.",
    "- A model comparison is incomplete if either model failed to run all 100 cases.",
    "",
    "## Artifacts",
    "",
    f"- Frozen cases: `{CASE_FILE.name}`",
    f"- Raw outputs and per-case scores: `{RESULT_FILE.name}`",
    f"- Machine-readable summary: `{SUMMARY_FILE.name}`",
]
REPORT_FILE.write_text("\n".join(lines) + "\n", encoding="utf-8")

print("\n===== C2 PILOT FINISHED =====")
print("Cases:", CASE_FILE)
print("Raw results:", RESULT_FILE)
print("Summary:", SUMMARY_FILE)
print("Report:", REPORT_FILE)
for mid, info in summary["models"].items():
    print(
        mid,
        "status=",
        info.get("status"),
        "results=",
        info.get("completed_results", 0),
        "error=",
        info.get("error"),
    )
print("\nNOTE: inspect model status and per-task N values before interpreting the comparison.")


Frozen dataset: /kaggle/working/pilot_cases_v2.jsonl
Dataset SHA-256: 910feb1905a3a9890c1ab0233f25c47e0cf26b94371b04740ff433a177221d69
Cases: 100 {'extraction': 20, 'strength': 20, 'json_validity': 20, 'tool_choice': 20, 'unsupported_edges': 20}

===== Loading Qwen/Qwen3-8B =====


Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

Loaded; running 100 cases.
Qwen/Qwen3-8B: 20/100 cases
Qwen/Qwen3-8B: 40/100 cases
Qwen/Qwen3-8B: 60/100 cases
Qwen/Qwen3-8B: 80/100 cases
Qwen/Qwen3-8B: 100/100 cases

===== Loading Qwen/Qwen3.5-9B =====


Loading weights:   0%|          | 0/760 [00:00<?, ?it/s]

Loaded; running 100 cases.
Qwen/Qwen3.5-9B: 20/100 cases
Qwen/Qwen3.5-9B: 40/100 cases
Qwen/Qwen3.5-9B: 60/100 cases
Qwen/Qwen3.5-9B: 80/100 cases
Qwen/Qwen3.5-9B: 100/100 cases

===== C2 PILOT FINISHED =====
Cases: /kaggle/working/pilot_cases_v2.jsonl
Raw results: /kaggle/working/pilot_results.jsonl
Summary: /kaggle/working/pilot_summary.json
Report: /kaggle/working/pilot_model.md
Qwen/Qwen3-8B status= completed results= 100 error= None
Qwen/Qwen3.5-9B status= completed results= 100 error= None

NOTE: inspect model status and per-task N values before interpreting the comparison.


In [15]:
import json
from pathlib import Path

p = Path("/kaggle/working/pilot_summary.json")
summary = json.loads(p.read_text(encoding="utf-8"))

for model_id, info in summary["models"].items():
    print(f"\n{'='*65}\n{model_id}")
    print("Status:", info.get("status"))
    print("Completed results:", info.get("completed_results"))

    for task, metrics in info.get("tasks", {}).items():
        print(f"\n{task} (n={metrics['n']})")
        for key in (
            "score",
            "strict_json_valid_rate",
            "schema_valid_rate",
            "exact_match_rate",
            "mean_latency_seconds",
            "errors",
        ):
            if key in metrics:
                print(f"  {key}: {metrics[key]}")

print("\n\n========== GENERATED REPORT ==========\n")
print(Path("/kaggle/working/pilot_model.md").read_text(encoding="utf-8"))



Qwen/Qwen3-8B
Status: completed
Completed results: 100

extraction (n=20)
  score: 0.15
  strict_json_valid_rate: 1.0
  schema_valid_rate: 1.0
  exact_match_rate: 0.15
  mean_latency_seconds: 4.8342600000000004
  errors: 0

strength (n=20)
  score: 0.7
  strict_json_valid_rate: 1.0
  schema_valid_rate: 1.0
  exact_match_rate: 0.7
  mean_latency_seconds: 1.6340150000000002
  errors: 0

json_validity (n=20)
  score: 1.0
  strict_json_valid_rate: 1.0
  schema_valid_rate: 1.0
  exact_match_rate: 1.0
  mean_latency_seconds: 2.4928850000000002
  errors: 0

tool_choice (n=20)
  score: 0.75
  strict_json_valid_rate: 1.0
  schema_valid_rate: 1.0
  exact_match_rate: 0.75
  mean_latency_seconds: 0.9174950000000001
  errors: 0

unsupported_edges (n=20)
  score: 1.0
  strict_json_valid_rate: 1.0
  schema_valid_rate: 1.0
  exact_match_rate: 1.0
  mean_latency_seconds: 1.915145
  errors: 0

Qwen/Qwen3.5-9B
Status: completed
Completed results: 100

extraction (n=20)
  score: 0.2
  strict_json_valid_r

In [16]:
import zipfile
from pathlib import Path

base = Path("/kaggle/working")
files = [
    "pilot_model.md",
    "pilot_cases_v2.jsonl",
    "pilot_results.jsonl",
    "pilot_summary.json",
]

zip_path = base / "c2_pilot_artifacts.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for name in files:
        path = base / name
        assert path.exists(), f"Missing artifact: {path}"
        z.write(path, arcname=name)

print("Created:", zip_path)
print("Size:", round(zip_path.stat().st_size / 1024, 1), "KB")


Created: /kaggle/working/c2_pilot_artifacts.zip
Size: 15.4 KB
